In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV, cross_val_predict

# Load datasets (adjust file paths as needed)
train_df = pd.read_csv('train_(2)_(1)_(1).csv')
dist_df = pd.read_csv('dist_from_city_centre_(1)_(1)_(1).csv')
rent_df = pd.read_csv('avg_rent_(1)_(1)_(1).csv')
test_df = pd.read_csv('test_(2)_(1)_(1).csv')

# Clean location names
for df in [train_df, test_df, dist_df, rent_df]:
    df['location'] = df['location'].str.strip().str.lower()

# Fix test data typos
test_df['location'] = test_df['location'].replace({'kanakapura  rod': 'kanakpura road', 'cooketown': 'cooke town'})

# Merge external data
train_df = train_df.merge(dist_df, on='location', how='left').merge(rent_df, on='location', how='left')
test_df = test_df.merge(dist_df, on='location', how='left').merge(rent_df, on='location', how='left')

# Impute missing values with median per location
for col in ['dist_from_city', 'avg_2bhk_rent']:
    train_df[col] = train_df.groupby('location')[col].transform(lambda x: x.fillna(x.median()))
    test_df[col] = test_df.groupby('location')[col].transform(lambda x: x.fillna(x.median()))


In [13]:
# Function to clean and convert total_sqft entries
def clean_total_sqft(sqft):
    try:
        sqft = str(sqft).strip()  # Convert to string and remove whitespace
        # Check if it's a pure number
        if sqft.replace('.', '').isdigit():
            return float(sqft)
        # Handle ranges like "1000-1200"
        if '-' in sqft:
            start, end = map(float, sqft.split('-'))
            return (start + end) / 2
        # Extract number and unit (if present)
        match = re.match(r'(\d+\.?\d*)\s*([A-Za-z\s\.]+)?', sqft)
        if match:
            value = float(match.group(1))
            unit = match.group(2).lower().replace(' ', '') if match.group(2) else ''
            # Conversion factors to square feet
            conversions = {
                'sq.meter': 10.7639,  # Square Meters to Sq. Ft.
                'sq.yards': 9,        # Square Yards to Sq. Ft.
                'acres': 43560,       # Acres to Sq. Ft.
                'perch': 272.25,      # Perch to Sq. Ft.
                'cents': 435.6,       # Cents to Sq. Ft.
                'guntha': 1089,       # Guntha to Sq. Ft.
                '': 1,                # No unit (already in Sq. Ft.)
                'sq.ft': 1            # Explicit Sq. Ft.
            }
            return value * conversions.get(unit, np.nan)
        return np.nan  # Return NaN for unparseable entries
    except:
        return np.nan

# Example usage with training and test datasets
# Assuming train_df and test_df are your DataFrames
train_df['total_sqft'] = train_df['total_sqft'].apply(clean_total_sqft)
test_df['total_sqft'] = test_df['total_sqft'].apply(clean_total_sqft)

# Impute missing values with the median from the training set
median_sqft = train_df['total_sqft'].median()
train_df['total_sqft'] = train_df['total_sqft'].fillna(median_sqft)
test_df['total_sqft'] = test_df['total_sqft'].fillna(median_sqft)

# Ensure the column is in float format
train_df['total_sqft'] = train_df['total_sqft'].astype(float)
test_df['total_sqft'] = test_df['total_sqft'].astype(float)

In [17]:
# Define the numeric columns to cap
numeric_cols = ['total_sqft', 'bath', 'balcony', 'avg_2bhk_rent', 'price']

# Calculate the 97.5th percentile for these columns in train_df
percentile_975 = train_df[numeric_cols].quantile(0.975)

# Cap outliers in train_df
for col in numeric_cols:
    train_df[col] = train_df[col].clip(upper=percentile_975[col])

# Cap outliers in test_df (excluding 'price' since it’s not in test_df)
test_numeric_cols = ['total_sqft', 'bath', 'balcony', 'avg_2bhk_rent']
for col in test_numeric_cols:
    test_df[col] = test_df[col].clip(upper=percentile_975[col])

In [19]:
# Standardize size
def standardize_size(size):
    try:
        size = str(size).strip().lower()
        match = re.match(r'(\d+)', size)
        if match:
            return f"{match.group(1)} BHK"
        if 'studio' in size:
            return "1 BHK"
        return np.nan
    except:
        return np.nan

train_df['size'] = train_df['size'].apply(standardize_size).fillna(train_df['size'].mode()[0])
test_df['size'] = test_df['size'].apply(standardize_size).fillna(train_df['size'].mode()[0])

# Impute bath and balcony based on size
size_bath_map = train_df.groupby('size')['bath'].median()
size_balcony_map = train_df.groupby('size')['balcony'].median()
train_df['bath'] = train_df.apply(lambda row: size_bath_map[row['size']] if pd.isna(row['bath']) else row['bath'], axis=1)
train_df['balcony'] = train_df.apply(lambda row: size_balcony_map[row['size']] if pd.isna(row['balcony']) else row['balcony'], axis=1)
test_df['bath'] = test_df.apply(lambda row: size_bath_map[row['size']] if pd.isna(row['bath']) else row['bath'], axis=1)
test_df['balcony'] = test_df.apply(lambda row: size_balcony_map[row['size']] if pd.isna(row['balcony']) else row['balcony'], axis=1)

# Simplify availability
train_df['availability'] = train_df['availability'].apply(lambda x: 'Ready' if x == 'Ready To Move' else 'Not Ready')
test_df['availability'] = test_df['availability'].apply(lambda x: 'Ready' if x == 'Ready To Move' else 'Not Ready')

# Feature engineering
train_df['location_zone'] = pd.cut(train_df['dist_from_city'], bins=[0, 5, 10, 15, np.inf], labels=['very_close', 'close', 'moderate', 'far'])
test_df['location_zone'] = pd.cut(test_df['dist_from_city'], bins=[0, 5, 10, 15, np.inf], labels=['very_close', 'close', 'moderate', 'far'])
train_df['sqft_zone_interaction'] = train_df['total_sqft'] * train_df['location_zone'].cat.codes
test_df['sqft_zone_interaction'] = test_df['total_sqft'] * test_df['location_zone'].cat.codes

# Target encoding for location
location_mean_price = train_df.groupby('location')['price'].mean()
train_df['location_encoded'] = train_df['location'].map(location_mean_price)
test_df['location_encoded'] = test_df['location'].map(location_mean_price).fillna(location_mean_price.mean())

# Drop unnecessary columns
train_df = train_df.drop(['ID', 'society', 'location'], axis=1)
test_ids = test_df['ID']
test_df = test_df.drop(['ID', 'society', 'location'], axis=1)

# Encode categorical variables
categorical_cols = ['area_type', 'availability', 'size', 'location_zone']
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Align test data columns
test_df = test_df.reindex(columns=train_df.columns.drop('price'), fill_value=0)

# Define features and target
X_train = train_df.drop('price', axis=1)
y_train = train_df['price']
X_test = test_df

# Log transform target
y_train_log = np.log1p(y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

KeyError: '18 BHK'